In [1]:
import os, json, warnings, re, glob
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, GridSearchCV, GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from quantile_forest import RandomForestQuantileRegressor

import shap

In [2]:
data_path = "/home/ji.song/blue/Planet_2023_GrazingSOC/1115final/RFSOC_v3_1020_reoder_v3_FesSta_FD2_withoutNewCattle_reorganize_1114final.csv"
whitelist_path = "/home/ji.song/blue/Planet_2023_GrazingSOC/1115final/dsm_cluster_vif_outputs/kept_vars.txt"

out_dir = "./rf_3nodes_CFval_kfold5_ESM3_cv_group_inner"
os.makedirs(out_dir, exist_ok=True)
os.makedirs(os.path.join(out_dir, "figs"), exist_ok=True)
os.makedirs(os.path.join(out_dir, "shap"), exist_ok=True)

target_col = "ESM 3 SOC (t/ha)"
random_state = 42

DO_SCALING = True
SCALER_TYPE = "standard"     # "standard" or "minmax"

param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [0.5, 0.7, 1.0]
}

OUTER_CV = 5
INNER_CV = 3

RFECV_STEP = 1
RFECV_MIN_FRAC = 0.10
RFECV_MIN_ABS  = 8

Q_LIST = [5,10,15,20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95]
Q_LOW_90, Q_HIGH_90 = 5, 95

SHAP_TOPK = 20

In [3]:
def rpiq(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    iqr  = np.subtract(*np.nanpercentile(y_true, [75, 25]))
    rmse = np.sqrt(np.nanmean((y_true - y_pred) ** 2))
    return (iqr / rmse) if rmse > 0 else np.inf


def pooled_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    rp = rpiq(y_true, y_pred)
    return r2, rmse, rp


def resolve_cols(wanted, actual_cols):
    amap = {c.lower(): c for c in actual_cols}
    resolved, missing = [], []
    for w in wanted:
        key = w.lower().strip()
        if key in amap:
            resolved.append(amap[key])
        else:
            missing.append(w)
    return resolved, missing


def make_scaler(scaler_type="standard"):
    return MinMaxScaler() if scaler_type == "minmax" else StandardScaler()


def apply_scaling_and_save(X_train, X_test, feature_names, save_json_path, scaler_type="standard"):
    scaler = make_scaler(scaler_type)
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    stats = {
        "scaler_type": scaler_type,
        "features": list(feature_names)
    }

    if scaler_type == "minmax":
        stats.update({
            "min_": scaler.min_.tolist(),
            "scale_": scaler.scale_.tolist(),
            "data_min_": scaler.data_min_.tolist(),
            "data_max_": scaler.data_max_.tolist(),
            "data_range_": scaler.data_range_.tolist()
        })
    else:
        stats.update({
            "mean_": scaler.mean_.tolist(),
            "scale_": scaler.scale_.tolist(),
            "var_": scaler.var_.tolist()
        })

    with open(save_json_path, "w", encoding="utf-8") as f:
        json.dump(stats, f, ensure_ascii=False, indent=2)

    return X_train_s, X_test_s


def make_group_cv(groups, desired_splits=3):
    groups = pd.Series(groups).astype(str).values
    n_groups = pd.Series(groups).nunique()

    if n_groups < 2:
        return None, None

    n_splits = min(desired_splits, n_groups)
    return GroupKFold(n_splits=n_splits), groups


def find_q_col(df, q_percent):
    c1, c2 = f"pre_{int(q_percent)}th", f"pre_{float(q_percent):.1f}th"
    for c in (c1, c2):
        if c in df.columns:
            return c

    pat = re.compile(rf"^pre_{q_percent}(?:\.0)?th$")
    for c in df.columns:
        if pat.match(c):
            return c

    raise ValueError(f"找不到分位列 {q_percent}%")

In [4]:
def _mean_encode_series(train_s: pd.Series, train_y: pd.Series, alpha=10.0, prior=None):
    if prior is None:
        prior = float(train_y.mean())

    stats = pd.DataFrame({
        "cat": train_s.values,
        "y": train_y.values
    })

    grp = stats.groupby("cat")["y"].agg(["mean", "count"])
    enc = (grp["count"] * grp["mean"] + alpha * prior) / (grp["count"] + alpha)

    return enc.to_dict(), prior


def target_encode_fit_transform_oof(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    cat_cols,
    n_splits=5,
    alpha=10.0,
    random_state=42,
    groups=None
):

    X_train = X_train.copy().reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    cat_cols = [c for c in cat_cols if c in X_train.columns]

    for c in cat_cols:
        X_train[c] = X_train[c].astype("category")

    num_cols = [c for c in X_train.columns if c not in cat_cols]

    X_train[num_cols] = X_train[num_cols].replace([np.inf, -np.inf], np.nan)
    train_medians = X_train[num_cols].median(numeric_only=True)
    X_train[num_cols] = X_train[num_cols].fillna(train_medians)

    global_prior = float(y_train.mean())

    for c in cat_cols:
        X_train[f"__te__{c}"] = np.nan

    use_group = groups is not None and pd.Series(groups).nunique() >= 2

    if use_group:
        groups = pd.Series(groups).astype(str).reset_index(drop=True).values
        n_unique_groups = pd.Series(groups).nunique()
        n_splits_eff = min(n_splits, n_unique_groups)

        splitter = GroupKFold(n_splits=n_splits_eff)
        split_iter = splitter.split(X_train, y_train, groups=groups)

        print(f"[TargetEncoding] Using GroupKFold: n_splits={n_splits_eff}, n_groups={n_unique_groups}")

    else:
        n_splits_eff = min(n_splits, len(X_train))
        splitter = KFold(n_splits=n_splits_eff, shuffle=True, random_state=random_state)
        split_iter = splitter.split(X_train)

        print(f"[TargetEncoding] Using KFold fallback: n_splits={n_splits_eff}")

    for tr_idx, va_idx in split_iter:
        tr_df, va_df = X_train.iloc[tr_idx], X_train.iloc[va_idx]
        tr_y = y_train.iloc[tr_idx]

        for c in cat_cols:
            mapping, prior = _mean_encode_series(
                tr_df[c],
                tr_y,
                alpha=alpha,
                prior=global_prior
            )

            vals = va_df[c].map(mapping).astype(float).fillna(prior)

            X_train.iloc[
                va_idx,
                X_train.columns.get_loc(f"__te__{c}")
            ] = vals.to_numpy()

    for c in cat_cols:
        X_train[f"__te__{c}"] = X_train[f"__te__{c}"].fillna(global_prior)

    encoders = {}

    for c in cat_cols:
        mapping, prior = _mean_encode_series(
            X_train[c],
            y_train,
            alpha=alpha,
            prior=global_prior
        )

        encoders[c] = {
            "mapping": mapping,
            "prior": prior,
            "alpha": alpha
        }

    te_cols = [f"__te__{c}" for c in cat_cols]
    enc_feature_names = list(num_cols) + te_cols

    X_train_enc = pd.concat(
        [
            X_train[num_cols].astype(float),
            X_train[te_cols].astype(float)
        ],
        axis=1
    ).values

    return X_train_enc, enc_feature_names, encoders, train_medians.to_dict(), cat_cols, num_cols


def target_encode_transform(X_df: pd.DataFrame, encoders, cat_cols, num_cols, train_medians: dict):
    X_df = X_df.copy()

    X_df[num_cols] = X_df[num_cols].replace([np.inf, -np.inf], np.nan)

    for c in num_cols:
        X_df[c] = X_df[c].fillna(train_medians.get(c, np.nanmedian(X_df[c])))

    te_cols = []

    for c in cat_cols:
        te_name = f"__te__{c}"
        te_cols.append(te_name)

        m = encoders[c]["mapping"]
        prior = encoders[c]["prior"]

        X_df[te_name] = X_df[c].map(m).astype(float).fillna(prior)

    X_mat = pd.concat(
        [
            X_df[num_cols].astype(float),
            X_df[te_cols].astype(float)
        ],
        axis=1
    ).values

    enc_feature_names = list(num_cols) + te_cols

    return X_mat, enc_feature_names

In [5]:
def plot_scatter(y_true, y_pred, title, out_png):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    lo = np.nanmin([np.nanmin(y_true), np.nanmin(y_pred)])
    hi = np.nanmax([np.nanmax(y_true), np.nanmax(y_pred)])

    pad = 0.05 * (hi - lo) if np.isfinite(hi - lo) and hi > lo else 1.0
    xmin, xmax = lo - pad, hi + pad

    r2, rmse, rp = pooled_metrics(y_true, y_pred)

    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    if mask.sum() >= 2:
        slope, intercept = np.polyfit(y_true[mask], y_pred[mask], 1)
    else:
        slope, intercept = np.nan, np.nan

    plt.figure(figsize=(5.5, 5.5), dpi=180)
    plt.scatter(y_true, y_pred, s=14, alpha=0.6)
    plt.plot([xmin, xmax], [xmin, xmax], linestyle="--")

    if np.isfinite(slope) and np.isfinite(intercept):
        xx = np.linspace(xmin, xmax, 100)
        plt.plot(xx, slope * xx + intercept)

    plt.xlim(xmin, xmax)
    plt.ylim(xmin, xmax)
    plt.gca().set_aspect("equal", adjustable="box")

    plt.xlabel("Observed (t/ha)")
    plt.ylabel("Predicted (t/ha)")
    plt.title(title)

    txt = f"R² = {r2:.3f}\nRMSE = {rmse:.3f}\nRPIQ = {rp:.3f}"

    if np.isfinite(slope) and np.isfinite(intercept):
        txt += f"\nŷ = {slope:.3f}·y + {intercept:.3f}"

    plt.gcf().text(0.62, 0.14, txt, ha="left", va="bottom")

    plt.tight_layout()
    plt.savefig(out_png, bbox_inches="tight")
    plt.close()


def compute_picp_curve(y, qdf, q_list=Q_LIST):
    y = np.asarray(y, dtype=float)
    out = []

    qs = sorted([q for q in q_list if 0 < q < 100])

    lo_idx, hi_idx = 0, len(qs) - 1

    while lo_idx < hi_idx:
        ql = qs[lo_idx]
        qh = qs[hi_idx]

        c_lo = qdf[f"pre_{ql}th"].to_numpy(float)
        c_hi = qdf[f"pre_{qh}th"].to_numpy(float)

        covered = np.mean((y >= c_lo) & (y <= c_hi))
        level = (qh - ql) / 100.0

        out.append({
            "PI_level": level,
            "PICP": covered,
            "q_low": ql,
            "q_high": qh
        })

        lo_idx += 1
        hi_idx -= 1

    return pd.DataFrame(out).sort_values("PI_level")


def plot_picp_multi(curves: dict, out_png, title="PICP vs PI level"):
    plt.figure(figsize=(8.5, 6.5), dpi=200)

    for name, dfc in curves.items():
        plt.plot(dfc["PI_level"], dfc["PICP"], lw=2.5, label=name)

        idx = int(np.argmin(np.abs(dfc["PI_level"].values - 0.90)))
        plt.scatter(
            [dfc["PI_level"].iloc[idx]],
            [dfc["PICP"].iloc[idx]],
            s=52
        )

    plt.plot([0, 1], [0, 1], ls="--", color="gray", lw=1)

    plt.xlabel("Nominal PI level")
    plt.ylabel("Empirical coverage (PICP)")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()


def boxplot_piwidth_multi(width_rows, ylabel, title, out_png):
    labels = [nm for nm, _ in width_rows]
    data = [np.asarray(w, float) for _, w in width_rows]

    plt.figure(figsize=(9, 6), dpi=200)
    plt.boxplot(data, labels=labels, showfliers=False)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(rotation=10)
    plt.tight_layout()
    plt.savefig(out_png)
    plt.close()

In [9]:
data = pd.read_csv(data_path)

data.columns = (
    data.columns
    .str.replace(r'[\u00A0\u2007\u202F]', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

if target_col not in data.columns:
    raise ValueError(f"缺少目标列: {target_col}")

y_all = data[target_col].astype(float)

COORD_X_COL = "Longitude"
COORD_Y_COL = "Latitude"

if (COORD_X_COL not in data.columns) or (COORD_Y_COL not in data.columns):
    raise ValueError(f"找不到坐标列 {COORD_X_COL}/{COORD_Y_COL}，请根据实际改列名")

GRID_DEG = 0.005

gx = np.floor(data[COORD_X_COL].astype(float) / GRID_DEG).astype("int64")
gy = np.floor(data[COORD_Y_COL].astype(float) / GRID_DEG).astype("int64")

data["grid1k_id"] = gx.astype(str) + "_" + gy.astype(str)

print("1km grids 总数:", data["grid1k_id"].nunique())

with open(whitelist_path, "r", encoding="utf-8") as f:
    whitelist = [ln.strip() for ln in f if ln.strip()]

whitelist = [c for c in whitelist if c in data.columns]

if not whitelist:
    raise ValueError("白名单与数据无交集")

user_cat_cols = [
    "Lookup_Soil_Taxnomy_Order",
    "cropLandcover_2008",
    "Annual_NLCD_LndCov_2008_CU_C1V1",
    "Level4_Ecoregions_FL",
    "Surficial_Geology_of_Florida",
    "Lookup_Hydrologic_Soil_Group",
]

present_cat, missing_cat = resolve_cols(user_cat_cols, data.columns)

print("[INFO] categorical columns found:", present_cat)
print("[INFO] categorical columns missing:", missing_cat)

feature_cols = list(dict.fromkeys(whitelist + present_cat))
X_full = data[feature_cols].copy()

for c in present_cat:
    if c in X_full.columns:
        X_full[c] = X_full[c].astype("category")

is_cf       = data["data_source"].astype(str).str.lower().eq("cattle farm")
is_flspd    = data["data_source"].astype(str).str.lower().eq("flspd")
is_graz     = data["Gtype_inter"].astype(str).str.lower().eq("grazing")
is_non_graz = data["Gtype_inter"].astype(str).str.lower().eq("no_graz")

idx_CF_graz     = np.where(is_cf & is_graz)[0]
idx_FLSPD_graz  = np.where(is_flspd & is_graz)[0]
idx_FLSPD_non   = np.where(is_flspd & is_non_graz)[0]

print("[INFO] CF_graz:", len(idx_CF_graz))
print("[INFO] FLSPD_graz:", len(idx_FLSPD_graz))
print("[INFO] FLSPD_non_graz:", len(idx_FLSPD_non))

if len(idx_CF_graz) < OUTER_CV:
    raise ValueError(f"CF∩grazing 样本过少：{len(idx_CF_graz)}，不足以 {OUTER_CV} 折。")

1km grids 总数: 982
[INFO] categorical columns found: ['Lookup_Soil_Taxnomy_Order', 'cropLandcover_2008', 'Annual_NLCD_LndCov_2008_CU_C1V1', 'Level4_Ecoregions_FL', 'Surficial_Geology_of_Florida', 'Lookup_Hydrologic_Soil_Group']
[INFO] categorical columns missing: []
[INFO] CF_graz: 75
[INFO] FLSPD_graz: 154
[INFO] FLSPD_non_graz: 790


In [12]:
fold_splits = []

grid_cf = data.loc[idx_CF_graz, "grid1k_id"].astype(str)
n_groups_cf = grid_cf.nunique()

print(f"[INFO] CF_graz: {len(idx_CF_graz)} samples, {n_groups_cf} unique 1km grids")

if n_groups_cf < 2:
    raise ValueError(f"1km grid group 数太少：{n_groups_cf}，无法做 block CV。")

outer_splits = min(OUTER_CV, n_groups_cf)

print(f"[INFO] Outer CV: GroupKFold by 1km grids, n_splits={outer_splits}")

outer_gkf = GroupKFold(n_splits=outer_splits)

pos = np.arange(len(idx_CF_graz))

for tr_pos, te_pos in outer_gkf.split(pos, groups=grid_cf.values):
    fold_splits.append((idx_CF_graz[tr_pos], idx_CF_graz[te_pos]))

print("[INFO] number of outer folds:", len(fold_splits))

[INFO] CF_graz: 75 samples, 58 unique 1km grids
[INFO] Outer CV: GroupKFold by 1km grids, n_splits=5
[INFO] number of outer folds: 5


In [13]:
def train_CF_only(cf_tr):
    return cf_tr


def train_CF_FLSPDgraz(cf_tr):
    return np.concatenate([cf_tr, idx_FLSPD_graz])


def train_CF_allFLSPD(cf_tr):
    return np.concatenate([cf_tr, idx_FLSPD_graz, idx_FLSPD_non])


nodes = {
    "CF_only": train_CF_only,
    "CFplus_FLSPDgraz": train_CF_FLSPDgraz,
    "CFplus_allFLSPD": train_CF_allFLSPD,
}

In [14]:
perfold_pred_paths = {
    name: os.path.join(out_dir, f"perfold_preds_{name}.csv")
    for name in nodes
}

perfold_q_paths = {
    name: os.path.join(out_dir, f"perfold_quantiles_{name}.csv")
    for name in nodes
}

for p in perfold_pred_paths.values():
    pd.DataFrame(
        columns=["node", "fold", "idx", "y_true", "y_pred"]
    ).to_csv(p, index=False)

for p in perfold_q_paths.values():
    pd.DataFrame(
        columns=["node", "fold", "idx", "y_true"]
    ).to_csv(p, index=False)

rfecv_summary_path = os.path.join(out_dir, "rfecv_summary.csv")

pd.DataFrame(
    columns=[
        "node",
        "fold",
        "n_features_total",
        "n_selected",
        "selected_features_json"
    ]
).to_csv(rfecv_summary_path, index=False)

In [15]:
results = []
all_node_fold_preds = []

node_picp_curves = {}
node_piwidth90 = {}
node_shap_meanabs = {}

for node_name, builder in nodes.items():

    print(f"\n===== Node: {node_name} =====")

    node_fold_preds = []
    node_fold_qdfs  = []
    shap_collect = {}

    for fold_id, (cf_tr, cf_te) in enumerate(fold_splits, 1):

        te_idx = cf_te
        tr_candidate = builder(cf_tr)

        # 关键：任何训练数据中，与当前 CF test grid 重叠的样本都删除
        te_grids = set(data.loc[te_idx, "grid1k_id"].astype(str).values)

        tr_filtered = [
            i for i in tr_candidate
            if str(data.at[i, "grid1k_id"]) not in te_grids
        ]

        tr_idx = np.array(tr_filtered, dtype=int)

        print(
            f"\n[{node_name} | Fold {fold_id}] "
            f"train candidate={len(tr_candidate)}, "
            f"after removing overlapping test grids={len(tr_idx)}, "
            f"test={len(te_idx)}"
        )

        X_train_raw = X_full.loc[tr_idx].copy()
        y_train = y_all.loc[tr_idx].copy()

        X_test_raw = X_full.loc[te_idx].copy()
        y_test = y_all.loc[te_idx].copy()

        grid_train = data.loc[tr_idx, "grid1k_id"].astype(str).values
        n_train_groups = pd.Series(grid_train).nunique()

        print(
            f"[{node_name} | Fold {fold_id}] "
            f"train samples={len(tr_idx)}, train grids={n_train_groups}"
        )

        # ============================================================
        # 1. Target encoding: inner OOF also uses grid-based GroupKFold
        # ============================================================

        X_train_te, enc_feature_names, encoders, train_medians, cat_cols, num_cols = target_encode_fit_transform_oof(
            X_train_raw,
            y_train,
            cat_cols=present_cat,
            n_splits=INNER_CV,
            alpha=10.0,
            random_state=random_state,
            groups=grid_train
        )

        X_test_te, _ = target_encode_transform(
            X_test_raw,
            encoders,
            cat_cols=cat_cols,
            num_cols=num_cols,
            train_medians=train_medians
        )

        # ============================================================
        # 2. Scaling: fit only on current outer training set
        # ============================================================

        if DO_SCALING:
            scaler_json = os.path.join(
                out_dir,
                f"scaler_{SCALER_TYPE}_{node_name}_fold{fold_id}.json"
            )

            X_train_used, X_test_used = apply_scaling_and_save(
                X_train_te,
                X_test_te,
                enc_feature_names,
                scaler_json,
                scaler_type=SCALER_TYPE
            )

        else:
            X_train_used, X_test_used = X_train_te, X_test_te

        # ============================================================
        # 3. RFECV: use grid-based GroupKFold inside the training fold
        # ============================================================

        n_total = X_train_used.shape[1]
        min_keep = max(
            int(np.ceil(n_total * RFECV_MIN_FRAC)),
            RFECV_MIN_ABS,
            1
        )

        inner_cv_obj, inner_groups = make_group_cv(
            grid_train,
            desired_splits=INNER_CV
        )

        if inner_cv_obj is None:
            raise ValueError(
                f"[{node_name} | Fold {fold_id}] "
                f"训练集 grid group 数量不足，无法做 GroupKFold。"
            )

        print(
            f"[{node_name} | Fold {fold_id}] "
            f"Inner GroupKFold for RFECV/GridSearch: "
            f"n_splits={inner_cv_obj.n_splits}, "
            f"n_groups={pd.Series(inner_groups).nunique()}"
        )

        base_rf = RandomForestRegressor(
            random_state=random_state,
            n_jobs=-1
        )

        rfecv = RFECV(
            estimator=base_rf,
            step=RFECV_STEP,
            cv=inner_cv_obj,
            scoring="neg_root_mean_squared_error",
            min_features_to_select=min_keep,
            n_jobs=-1
        )

        rfecv.fit(
            X_train_used,
            y_train,
            groups=inner_groups
        )

        support_mask = rfecv.support_
        selected_names = list(np.array(enc_feature_names)[support_mask])
        n_selected = int(support_mask.sum())

        print(
            f"[{node_name} | Fold {fold_id}] "
            f"RFECV selected {n_selected}/{n_total} features "
            f"(min_keep={min_keep})"
        )

        with open(
            os.path.join(out_dir, f"rfecv_selected_{node_name}_fold{fold_id}.txt"),
            "w",
            encoding="utf-8"
        ) as ftxt:
            ftxt.write("\n".join(selected_names))

        pd.DataFrame([{
            "node": node_name,
            "fold": fold_id,
            "n_features_total": n_total,
            "n_selected": n_selected,
            "selected_features_json": json.dumps(selected_names, ensure_ascii=False)
        }]).to_csv(
            rfecv_summary_path,
            mode="a",
            header=False,
            index=False
        )

        X_train_sel = X_train_used[:, support_mask]
        X_test_sel  = X_test_used[:, support_mask]

        # ============================================================
        # 4. GridSearchCV: also use grid-based GroupKFold
        # ============================================================

        grid_cv_obj, grid_groups = make_group_cv(
            grid_train,
            desired_splits=INNER_CV
        )

        grid = GridSearchCV(
            RandomForestRegressor(
                random_state=random_state,
                n_jobs=-1
            ),
            param_grid=param_grid,
            cv=grid_cv_obj,
            n_jobs=-1,
            scoring="neg_root_mean_squared_error",
            refit=True
        )

        grid.fit(
            X_train_sel,
            y_train,
            groups=grid_groups
        )

        best = grid.best_estimator_

        print(
            f"[{node_name} | Fold {fold_id}] "
            f"Best params: {grid.best_params_}"
        )

        # ============================================================
        # 5. Prediction and metrics
        # ============================================================

        y_pred_train = best.predict(X_train_sel)
        y_pred = best.predict(X_test_sel)

        train_r2 = r2_score(y_train, y_pred_train)
        train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
        train_rp = rpiq(y_train, y_pred_train)

        test_r2 = r2_score(y_test, y_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        test_rp = rpiq(y_test, y_pred)

        print(
            f"[{node_name} | Fold {fold_id}] "
            f"Train R²={train_r2:.3f} RMSE={train_rmse:.3f} RPIQ={train_rp:.3f} | "
            f"Test R²={test_r2:.3f} RMSE={test_rmse:.3f} RPIQ={test_rp:.3f}"
        )

        results.append({
            "node": node_name,
            "fold": fold_id,
            "r2": float(test_r2),
            "rmse": float(test_rmse),
            "rpiq": float(test_rp),
            "train_r2": float(train_r2),
            "train_rmse": float(train_rmse),
            "train_rpiq": float(train_rp),
            "train_size": int(len(tr_idx)),
            "test_size": int(len(te_idx)),
            "train_grid_count": int(n_train_groups),
            "test_grid_count": int(pd.Series(data.loc[te_idx, "grid1k_id"]).nunique()),
            "best_params": json.dumps(grid.best_params_),
            "n_features_total": int(n_total),
            "n_selected": int(n_selected),
            "scaled": int(DO_SCALING),
            "scaler_type": SCALER_TYPE,
            "inner_cv": "GroupKFold_by_grid1k_id",
            "inner_cv_splits": int(grid_cv_obj.n_splits)
        })

        fold_pred_df = pd.DataFrame({
            "node": node_name,
            "fold": fold_id,
            "idx": te_idx.astype(int),
            "y_true": y_test.values,
            "y_pred": y_pred
        })

        fold_pred_df.to_csv(
            perfold_pred_paths[node_name],
            mode="a",
            header=False,
            index=False
        )

        node_fold_preds.append(fold_pred_df)
        all_node_fold_preds.append(fold_pred_df)

        # ============================================================
        # 6. Quantile Random Forest
        # ============================================================

        qrf = RandomForestQuantileRegressor(
            random_state=random_state,
            n_estimators=grid.best_params_.get("n_estimators", 400),
            max_depth=grid.best_params_.get("max_depth", None),
            min_samples_split=grid.best_params_.get("min_samples_split", 2),
            min_samples_leaf=grid.best_params_.get("min_samples_leaf", 1),
            max_features=grid.best_params_.get("max_features", 1.0),
            n_jobs=-1
        )

        qrf.fit(X_train_sel, y_train)

        q_levels = [q / 100.0 for q in Q_LIST]
        q_mat = qrf.predict(X_test_sel, quantiles=q_levels)

        q_cols = [f"pre_{q}th" for q in Q_LIST]

        fold_q_df = pd.DataFrame(q_mat, columns=q_cols)

        fold_q_df.insert(0, "y_true", y_test.values)
        fold_q_df.insert(0, "idx", te_idx.astype(int))
        fold_q_df.insert(0, "fold", fold_id)
        fold_q_df.insert(0, "node", node_name)

        fold_q_df["PI90_width"] = (
            fold_q_df[f"pre_{Q_HIGH_90}th"] -
            fold_q_df[f"pre_{Q_LOW_90}th"]
        )

        cover90 = (
            (fold_q_df["y_true"] >= fold_q_df[f"pre_{Q_LOW_90}th"]) &
            (fold_q_df["y_true"] <= fold_q_df[f"pre_{Q_HIGH_90}th"])
        )

        fold_q_df["cover90"] = cover90.astype(int)

        fold_q_df.to_csv(
            perfold_q_paths[node_name],
            mode="a",
            header=False,
            index=False
        )

        node_fold_qdfs.append(fold_q_df)

        # ============================================================
        # 7. SHAP
        # ============================================================

        try:
            explainer = shap.TreeExplainer(
                best,
                feature_names=selected_names
            )

            shap_vals = explainer.shap_values(X_test_sel)
            base_val = explainer.expected_value

            mean_abs = np.abs(shap_vals).mean(axis=0)

            shap_fold_df = pd.DataFrame({
                "feature": selected_names,
                "mean_abs_shap": mean_abs,
                "fold": fold_id,
                "node": node_name
            }).sort_values("mean_abs_shap", ascending=False)

            shap_fold_csv = os.path.join(
                out_dir,
                "shap",
                f"shap_meanabs_{node_name}_fold{fold_id}.csv"
            )

            shap_fold_df.to_csv(shap_fold_csv, index=False)

            w = len(te_idx)

            for feat, val in zip(selected_names, mean_abs):
                shap_collect[feat] = shap_collect.get(feat, 0.0) + float(val) * w

            topk_idx = np.argsort(-mean_abs)[:SHAP_TOPK]

            exp = shap.Explanation(
                values=shap_vals[:, topk_idx],
                base_values=np.full(len(X_test_sel), base_val),
                data=X_test_sel[:, topk_idx],
                feature_names=[selected_names[i] for i in topk_idx]
            )

            plt.figure(figsize=(9, 6.5), dpi=200)
            shap.plots.beeswarm(
                exp,
                max_display=SHAP_TOPK,
                show=False
            )

            plt.title(
                f"SHAP Beeswarm — {node_name} (fold {fold_id})",
                fontsize=11
            )

            shp_png = os.path.join(
                out_dir,
                "shap",
                f"shap_beeswarm_{node_name}_fold{fold_id}.png"
            )

            plt.tight_layout()
            plt.savefig(shp_png)
            plt.close()

        except Exception as e:
            print(f"[WARN] SHAP failed: {node_name} fold {fold_id}: {e}")

    # ============================================================
    # Node-level pooled results
    # ============================================================

    if node_fold_preds:
        pooled_df = pd.concat(node_fold_preds, ignore_index=True)

        pooled_df.to_csv(
            os.path.join(out_dir, f"pooled_{node_name}.csv"),
            index=False
        )

        r2p, rmsep, rpip = pooled_metrics(
            pooled_df["y_true"].values,
            pooled_df["y_pred"].values
        )

        pooled_sum_path = os.path.join(
            out_dir,
            "pooled_summary_by_node.csv"
        )

        row = pd.DataFrame([{
            "node": node_name,
            "pooled_r2": r2p,
            "pooled_rmse": rmsep,
            "pooled_rpiq": rpip,
            "n_points": int(len(pooled_df))
        }])

        if os.path.exists(pooled_sum_path):
            old = pd.read_csv(pooled_sum_path)
            old = old[old["node"] != node_name]
            new = pd.concat([old, row], ignore_index=True)
            new.to_csv(pooled_sum_path, index=False)
        else:
            row.to_csv(pooled_sum_path, index=False)

        png_path = os.path.join(
            out_dir,
            "figs",
            f"scatter_{node_name}.png"
        )

        plot_scatter(
            pooled_df["y_true"].values,
            pooled_df["y_pred"].values,
            title=f"{node_name} — pooled grid-based 5-fold CF test",
            out_png=png_path
        )

        print(
            f"📈 [{node_name}] pooled Test: "
            f"R²={r2p:.3f} RMSE={rmsep:.3f} RPIQ={rpip:.3f}"
        )

    # ============================================================
    # Node-level uncertainty results
    # ============================================================

    if node_fold_qdfs:
        pooled_q = pd.concat(node_fold_qdfs, ignore_index=True)

        pooled_q.to_csv(
            os.path.join(out_dir, f"pooled_quantiles_{node_name}.csv"),
            index=False
        )

        curve = compute_picp_curve(
            pooled_q["y_true"].values,
            pooled_q,
            q_list=Q_LIST
        )

        node_picp_curves[node_name] = curve

        curve_csv = os.path.join(
            out_dir,
            f"picp_curve_points_{node_name}.csv"
        )

        curve.to_csv(curve_csv, index=False)

        idx_090 = int(
            np.argmin(np.abs(curve["PI_level"].values - 0.90))
        )

        picp90 = float(curve["PICP"].iloc[idx_090])

        with open(
            os.path.join(out_dir, f"picp90_{node_name}.txt"),
            "w"
        ) as f:
            f.write(f"PICP@0.90 = {picp90:.6f}\n")

        w90 = (
            pooled_q[f"pre_{Q_HIGH_90}th"] -
            pooled_q[f"pre_{Q_LOW_90}th"]
        ).to_numpy(float)

        node_piwidth90[node_name] = w90

        pd.DataFrame({
            "PI90_width": w90
        }).to_csv(
            os.path.join(out_dir, f"piwidth90_{node_name}.csv"),
            index=False
        )

        plot_picp_multi(
            {node_name: curve},
            out_png=os.path.join(
                out_dir,
                "figs",
                f"picp_curve_{node_name}.png"
            ),
            title=f"PICP vs PI level — {node_name} CF test"
        )

    # ============================================================
    # Node-level SHAP aggregation
    # ============================================================

    if shap_collect:
        total_w = sum(
            [len(df) for df in node_fold_preds]
        ) if node_fold_preds else 1

        agg_shap = pd.DataFrame({
            "feature": list(shap_collect.keys()),
            "mean_abs_shap_weighted": [
                v / total_w for v in shap_collect.values()
            ]
        })

        agg_shap = agg_shap.sort_values(
            "mean_abs_shap_weighted",
            ascending=False
        )

        node_shap_meanabs[node_name] = agg_shap

        agg_shap.to_csv(
            os.path.join(
                out_dir,
                "shap",
                f"shap_meanabs_aggregated_{node_name}.csv"
            ),
            index=False
        )

        top = agg_shap.head(SHAP_TOPK)

        plt.figure(figsize=(7.6, 6), dpi=200)
        plt.barh(
            top["feature"][::-1],
            top["mean_abs_shap_weighted"][::-1]
        )

        plt.xlabel("mean(|SHAP|) weighted by test size")
        plt.title(f"Top-{SHAP_TOPK} features by mean|SHAP| — {node_name}")
        plt.tight_layout()

        plt.savefig(
            os.path.join(
                out_dir,
                "shap",
                f"shap_top{SHAP_TOPK}_bar_{node_name}.png"
            )
        )

        plt.close()


===== Node: CF_only =====

[CF_only | Fold 1] train candidate=60, after removing overlapping test grids=60, test=15
[CF_only | Fold 1] train samples=60, train grids=47
[TargetEncoding] Using GroupKFold: n_splits=3, n_groups=47
[CF_only | Fold 1] Inner GroupKFold for RFECV/GridSearch: n_splits=3, n_groups=47
[CF_only | Fold 1] RFECV selected 178/217 features (min_keep=22)
[CF_only | Fold 1] Best params: {'max_depth': 10, 'max_features': 0.7, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
[CF_only | Fold 1] Train R²=0.861 RMSE=4.657 RPIQ=1.360 | Test R²=0.372 RMSE=7.989 RPIQ=1.175

[CF_only | Fold 2] train candidate=60, after removing overlapping test grids=60, test=15
[CF_only | Fold 2] train samples=60, train grids=47
[TargetEncoding] Using GroupKFold: n_splits=3, n_groups=47
[CF_only | Fold 2] Inner GroupKFold for RFECV/GridSearch: n_splits=3, n_groups=47
[CF_only | Fold 2] RFECV selected 22/217 features (min_keep=22)
[CF_only | Fold 2] Best params: {'max_depth':

In [16]:
res_df = pd.DataFrame(results)

res_df.to_csv(
    os.path.join(out_dir, "rf_3nodes_results.csv"),
    index=False
)

agg = res_df.groupby("node", as_index=False).agg(
    r2_mean=("r2", "mean"),
    r2_std=("r2", "std"),
    rmse_mean=("rmse", "mean"),
    rmse_std=("rmse", "std"),
    rpiq_mean=("rpiq", "mean"),
    rpiq_std=("rpiq", "std"),
    train_r2_mean=("train_r2", "mean"),
    train_r2_std=("train_r2", "std"),
    train_rmse_mean=("train_rmse", "mean"),
    train_rmse_std=("train_rmse", "std"),
    train_rpiq_mean=("train_rpiq", "mean"),
    train_rpiq_std=("train_rpiq", "std"),
    train_size_mean=("train_size", "mean"),
    test_size_mean=("test_size", "mean"),
    train_grid_count_mean=("train_grid_count", "mean"),
    test_grid_count_mean=("test_grid_count", "mean"),
    n_features_total_mean=("n_features_total", "mean"),
    n_selected_mean=("n_selected", "mean"),
    scaled_mean=("scaled", "mean")
)

agg.to_csv(
    os.path.join(out_dir, "rf_3nodes_summary.csv"),
    index=False
)

if all_node_fold_preds:
    pd.concat(
        all_node_fold_preds,
        ignore_index=True
    ).to_csv(
        os.path.join(out_dir, "perfold_preds_all_nodes.csv"),
        index=False
    )

print("\n✅ Done. Per-node artifacts saved.")


✅ Done. Per-node artifacts saved.


In [17]:
# Cross-node PICP curves

if node_picp_curves:
    out_picp_all = os.path.join(
        out_dir,
        "figs",
        "picp_curve_ALL_nodes.png"
    )

    plot_picp_multi(
        node_picp_curves,
        out_png=out_picp_all,
        title="PICP vs PI level — ALL nodes, grid-based CF test"
    )

    rows = []

    for nm, dfc in node_picp_curves.items():
        tmp = dfc.copy()
        tmp["node"] = nm
        rows.append(tmp)

    pd.concat(
        rows,
        ignore_index=True
    ).to_csv(
        os.path.join(out_dir, "picp_curve_points_ALL_nodes.csv"),
        index=False
    )

# 90% PI width boxplot across nodes

if node_piwidth90:
    rows = [
        (nm, vals)
        for nm, vals in node_piwidth90.items()
    ]

    out_pi_all = os.path.join(
        out_dir,
        "figs",
        "piwidth90_box_ALL_nodes.png"
    )

    boxplot_piwidth_multi(
        rows,
        ylabel="PI Width, 5%–95%",
        title="90% Prediction Interval Width — ALL nodes, CF test",
        out_png=out_pi_all
    )

    stats = []

    for nm, v in node_piwidth90.items():
        a = np.asarray(v, float)

        stats.append({
            "node": nm,
            "count": a.size,
            "mean": float(np.mean(a)),
            "median": float(np.median(a)),
            "std": float(np.std(a, ddof=1) if a.size > 1 else 0.0),
            "min": float(np.min(a)),
            "max": float(np.max(a))
        })

    pd.DataFrame(stats).to_csv(
        os.path.join(out_dir, "piwidth90_summary_ALL_nodes.csv"),
        index=False
    )

print("📦 Cross-node figures saved under:", os.path.join(out_dir, "figs"))

📦 Cross-node figures saved under: ./rf_3nodes_CFval_kfold5_ESM3_cv_group_inner/figs


In [18]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE_DIR = "./rf_3nodes_CFval_kfold5_ESM3_cv_group_inner"

NODES = [
    "CF_only",
    "CFplus_FLSPDgraz",
    "CFplus_allFLSPD",
]


def plot_final_scatter(node_name, base_dir=BASE_DIR, dpi=1000):

    pooled_path = os.path.join(base_dir, f"pooled_{node_name}.csv")

    if not os.path.exists(pooled_path):
        raise FileNotFoundError(f"找不到文件: {pooled_path}")

    df = pd.read_csv(pooled_path)

    y_true = df["y_true"].astype(float).values
    y_pred = df["y_pred"].astype(float).values

    x_min, x_max = np.nanmin(y_true), np.nanmax(y_true)
    y_min, y_max = np.nanmin(y_pred), np.nanmax(y_pred)

    data_min = min(x_min, y_min)
    data_max = max(x_max, y_max)

    if np.isfinite(data_max - data_min) and data_max > data_min:
        pad = 0.05 * (data_max - data_min)
    else:
        pad = 1.0

    lo = data_min - pad
    hi = data_max + pad

    mask = np.isfinite(y_true) & np.isfinite(y_pred)

    if mask.sum() >= 2:
        slope, intercept = np.polyfit(y_true[mask], y_pred[mask], 1)
        line_x = np.linspace(x_min, x_max, 200)
        line_y = slope * line_x + intercept
    else:
        line_x = None
        line_y = None

    plt.figure(figsize=(5.5, 5.5), dpi=dpi)

    plt.scatter(y_true, y_pred, s=10, alpha=0.6)

    plt.plot(
        [lo, hi],
        [lo, hi],
        linestyle="--",
        color="black",
        linewidth=1.0
    )

    if line_x is not None:
        plt.plot(
            line_x,
            line_y,
            color="orange",
            linewidth=1.5
        )

    plt.xlim(lo, hi)
    plt.ylim(lo, hi)
    plt.gca().set_aspect("equal", adjustable="box")

    plt.grid(True, linestyle="--", alpha=0.3)

    plt.xlabel("Observed SOC stocks (t/ha)")
    plt.ylabel("Predicted SOC stocks (t/ha)")
    plt.title(node_name)

    plt.tight_layout()

    out_png = os.path.join(
        base_dir,
        "figs",
        f"ESM3_scatter_{node_name}_1000dpi.png"
    )

    plt.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.close()

    print(f"[OK] Saved: {out_png}")


os.makedirs(os.path.join(BASE_DIR, "figs"), exist_ok=True)

for node in NODES:
    plot_final_scatter(node)

[OK] Saved: ./rf_3nodes_CFval_kfold5_ESM3_cv_group_inner/figs/ESM3_scatter_CF_only_1000dpi.png
[OK] Saved: ./rf_3nodes_CFval_kfold5_ESM3_cv_group_inner/figs/ESM3_scatter_CFplus_FLSPDgraz_1000dpi.png
[OK] Saved: ./rf_3nodes_CFval_kfold5_ESM3_cv_group_inner/figs/ESM3_scatter_CFplus_allFLSPD_1000dpi.png
